# 03 — Random Forest baseline: RGB vs 13 spectral bands

This notebook trains and evaluates the classical baseline using the **unchanged spatial split and training-only band statistics** saved by notebook 02.

**Functions:** `src/piece_of_crop/features.py` (`extract_rf_features`), `data.py` (`read_split`, `get_cache`) and `train.py` (`predictions_table`, the shared format for saved predictions).

**Outputs:** six per-image test prediction CSVs under `results/predictions/`, model metrics in `results/rf_results.csv`, and an RGB/13-band comparison chart under `results/figures/`.

**Run location:** the project's `notebooks/` folder, with the `piece-of-crop` environment as kernel and the package installed (`python -m pip install -e .`, see README). The raw EuroSAT files or an existing `data/eurosat_cache.npy` are required.

## 1. Setup and shared data from notebook 02

All functions come from the installed `piece_of_crop` package, like in notebooks 01 and 02. `find_project_root` finds the project folder whether the notebook runs from the project root or from `notebooks/`.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

from piece_of_crop import CLASS_NAMES, extract_rf_features, find_project_root, get_cache, predictions_table, read_split

SEEDS = (0, 1, 2)

project_root = find_project_root()
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"
predictions_dir = results_dir / "predictions"
models_dir = project_root / "models"
for folder in (results_dir, figures_dir, predictions_dir, models_dir):
    folder.mkdir(parents=True, exist_ok=True)

split = read_split(results_dir / "split.csv")
band_stats = pd.read_csv(results_dir / "band_stats.csv")
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")

print("Project root:", project_root)
print("Patches per split:", split["split"].value_counts().to_dict())
print("Cache shape:", cache.shape)

## 2. Extract tabular features

For **RGB**, calculate mean and standard deviation of bands B02, B03 and B04 (six features). For **13-band**, calculate the same two statistics for all 13 channels plus NDVI mean and standard deviation (28 features). Both come from exactly the same source images and training-only normalization parameters.


In [ ]:
X_rgb, X_ms, rgb_feature_names, ms_feature_names = extract_rf_features(cache, band_stats)

assert X_rgb.shape == (len(split), 6)
assert X_ms.shape == (len(split), 28)
assert np.isfinite(X_rgb).all() and np.isfinite(X_ms).all()

print("RGB feature matrix:", X_rgb.shape)
print("13-band feature matrix:", X_ms.shape)

train_mask = split["split"].eq("train").to_numpy()
val_mask = split["split"].eq("val").to_numpy()
test_mask = split["split"].eq("test").to_numpy()
y = split["label"].to_numpy(dtype=int)
assert train_mask.sum() and val_mask.sum() and test_mask.sum()


## 3. Fit Random Forest baselines and export test predictions

Fit one model per input type and random seed, with the same hyperparameters and identical train/validation/test splits. We report macro-F1 and accuracy for each split. There is no hyperparameter search or test-set-driven tuning; the fixed hyperparameters below are a transparent baseline.

Every test CSV has `file`, `y_true`, `y_pred` and `confidence`, plus class probabilities (`p_0` ... `p_9`) for potential later human-review analysis. These files are inputs to the team's confusion matrices and cost-weighted error analysis.


In [ ]:
rows = []
trained_models = {}
label_ids = np.sort(split["label"].unique())
assert len(label_ids) == len(CLASS_NAMES), "Expected 10 classes from notebook 02"

for input_type, X in (("rgb", X_rgb), ("13band", X_ms)):
    for seed in SEEDS:
        print(f"Training Random Forest: {input_type}, seed={seed}")
        model = RandomForestClassifier(
            n_estimators=200,
            max_features="sqrt",
            random_state=seed,
            n_jobs=-1,
        )
        model.fit(X[train_mask], y[train_mask])
        trained_models[(input_type, seed)] = model
        joblib.dump(model, models_dir / f"rf_{input_type}_seed{seed}.joblib")

        for subset, mask in (("val", val_mask), ("test", test_mask)):
            y_true = y[mask]
            predicted = model.predict(X[mask])
            prob = model.predict_proba(X[mask])
            rows.append({
                "model": "RandomForest",
                "input": input_type,
                "seed": seed,
                "split": subset,
                "accuracy": accuracy_score(y_true, predicted),
                "macro_f1": f1_score(y_true, predicted, average="macro"),
                "n_samples": int(mask.sum()),
            })

            if subset == "test":
                # predict_proba columns follow model.classes_, which must be 0..9 in order.
                assert np.array_equal(model.classes_, label_ids)
                out = predictions_table(split.loc[mask, "file"], y_true, prob)
                output_path = predictions_dir / f"rf_{input_type}_seed{seed}.csv"
                out.to_csv(output_path, index=False)
                print("Saved", output_path.name, "rows:", len(out))

results_df = pd.DataFrame(rows)
results_df.to_csv(results_dir / "rf_results.csv", index=False)
display(results_df)

## 4. Summarize RGB vs 13-band performance

Aggregate the three seeds, then show a per-class report for seed 0. Results here are generated by running the notebook; no accuracy values are assumed in advance.


In [ ]:
import matplotlib.pyplot as plt

comparison = results_df.groupby(["split", "input"])[["accuracy", "macro_f1"]].agg(["mean", "std"])
print("Validation and test performance across 3 random seeds")
display(comparison.round(4))

test_results = results_df[results_df["split"].eq("test")]
mean_scores = test_results.groupby("input")[["accuracy", "macro_f1"]].mean()
ax = mean_scores.reindex(["rgb", "13band"]).plot(kind="bar", figsize=(8, 5), ylim=(0, 1), rot=0)
ax.set(title="Random Forest: RGB vs 13 bands (test)", xlabel="Input features", ylabel="Score")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / "rf_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

for input_type in ("rgb", "13band"):
    pred = pd.read_csv(predictions_dir / f"rf_{input_type}_seed0.csv")
    print(f"\nPer-class report: {input_type} (seed 0)")
    print(classification_report(
        pred["y_true"], pred["y_pred"],
        labels=label_ids, target_names=list(CLASS_NAMES), zero_division=0,
    ))

print("Done. Predictions:", predictions_dir)
print("Metrics:", results_dir / "rf_results.csv")
print("Plot:", figures_dir / "rf_comparison.png")
